# 节点13：Chinchilla 缩放定律（2022）

本 notebook 用纯 NumPy 手撕 Chinchilla 缩放定律的核心数学：
- 幂律损失函数
- 给定计算预算，如何找最优参数量和数据量
- 用数字验证 Chinchilla (70B) 为何优于 Gopher (280B)

> 目标读者：会基础代数 + 基础 Python 的初中生

In [ ]:
import numpy as np
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
import os

np.random.seed(42)
ASSETS = '../docs/assets'
os.makedirs(ASSETS, exist_ok=True)

print('准备就绪，开始探索缩放定律！')

## Part 1：幂律是什么感觉？

语言模型的损失值随参数量增加而下降，遵循幂律：

```
L(N) ≈ A / N^α
```

让我们用数字感受一下这个规律。

In [ ]:
# 幂律演示：L(N) = A / N^alpha
# 用 Kaplan 2020 论文里估计的 alpha ≈ 0.076 作为示意
A = 7.0   # 常数，和任务难度有关
alpha = 0.076  # 幂指数（Kaplan 2020 的估计）

# 参数量从 10M 到 500B
N_values = np.logspace(7, 11.7, 100)  # 10^7 到 5×10^11

def power_law_loss(N, A, alpha):
    """单因子幂律损失：L = A / N^alpha"""
    return A / (N ** alpha)

L_values = power_law_loss(N_values, A, alpha)

fig, axes = plt.subplots(1, 2, figsize=(12, 4))

# 左图：普通坐标
axes[0].plot(N_values / 1e9, L_values, 'b-', linewidth=2)
axes[0].set_xlabel('参数量 (Billions)')
axes[0].set_ylabel('损失值 L')
axes[0].set_title('普通坐标：看起来很快饱和')
axes[0].grid(True, alpha=0.3)

# 右图：log-log 坐标（幂律在 log-log 下是直线）
axes[1].loglog(N_values, L_values, 'r-', linewidth=2)
axes[1].set_xlabel('参数量 N（log 坐标）')
axes[1].set_ylabel('损失值 L（log 坐标）')
axes[1].set_title('Log-Log 坐标：幂律是一条直线！')
axes[1].grid(True, alpha=0.3, which='both')

plt.tight_layout()
plt.savefig(f'{ASSETS}/13-power-law.png', dpi=100, bbox_inches='tight')
plt.close()

# 验算：参数量增加 10 倍，损失变化多少？
N1, N2 = 1e9, 10e9
ratio = power_law_loss(N1, A, alpha) / power_law_loss(N2, A, alpha)
print(f'参数量从 1B → 10B（增加 10 倍），损失比值 = {ratio:.4f}')
print(f'10^alpha = 10^{alpha} = {10**alpha:.4f}')
print(f'所以损失只降低到原来的 1/{ratio:.2f} ≈ {1/ratio:.0f}%')
print(f'\n规律：参数量增加 10 倍，损失只降低 {(1-1/ratio)*100:.1f}%（收益递减）')

## Part 2：双因子损失函数

真实损失取决于两件事：参数量（N）和数据量（D）。

```
L(N, D) = A / N^α  +  B / D^β  +  L_∞
```

- `A / N^α`：模型太小带来的损失
- `B / D^β`：数据太少带来的损失
- `L_∞ ≈ 1.69`：语言本身的不确定性（不可减少）

我们用 Hoffmann et al. (2022) 论文中的估计参数演示。

In [ ]:
# Chinchilla 论文的参数（Hoffmann et al. 2022，表A1，近似值）
A_param = 406.4
B_param = 410.7
alpha_c = 0.34   # 参数量的幂指数
beta_c  = 0.28   # 数据量的幂指数
L_inf   = 1.69   # 最小理论损失（不可减少的熵）

def chinchilla_loss(N, D):
    """双因子损失：L(N,D) = A/N^alpha + B/D^beta + L_inf"""
    return A_param / (N ** alpha_c) + B_param / (D ** beta_c) + L_inf

# 用几组具体数字感受一下
examples = [
    ('GPT-3 (175B, 300B tok)',   175e9,  300e9),
    ('Gopher (280B, 300B tok)',  280e9,  300e9),
    ('Chinchilla (70B, 1.4T tok)', 70e9, 1400e9),
    ('LLaMA-7B (7B, 1T tok)',      7e9, 1000e9),
]

print(f'{'模型':<30} {'N':>12} {'D':>14} {'预测损失':>10}')
print('-' * 70)
for name, N, D in examples:
    L = chinchilla_loss(N, D)
    print(f'{name:<30} {N/1e9:>10.0f}B {D/1e9:>12.0f}B  {L:>10.4f}')

print('\n注意：损失越低越好。Chinchilla 70B 的损失比 Gopher 280B 更低！')

## Part 3：给定计算量，如何找最优 N 和 D？

训练大模型的计算量大约是：

```
C ≈ 6 × N × D   （FLOPs，每参数每token约6次浮点运算）
```

**问题**：给定固定的 C，怎么分配 N 和 D 使 L(N,D) 最小？

我们用**网格搜索**（Grid Search）来数值验证这个最优分配。

In [ ]:
def optimal_allocation(C_flops):
    """给定计算预算 C，用网格搜索找最优 (N, D)"""
    best_loss = float('inf')
    best_N = None
    best_D = None
    
    # 搜索 N：从 1B 到 C/6B（不超过全部预算都用来做参数）
    # C = 6*N*D，所以 D = C/(6*N)，D 至少要有 1M token
    N_candidates = np.logspace(8, np.log10(C_flops / 6 / 1e6), 200)
    
    for N in N_candidates:
        D = C_flops / (6 * N)  # 固定 C，由 N 推出 D
        if D < 1e6:  # D 至少要有 100 万 token
            continue
        L = chinchilla_loss(N, D)
        if L < best_loss:
            best_loss = L
            best_N = N
            best_D = D
    
    return best_N, best_D, best_loss

# 对比不同计算预算下的最优分配
compute_budgets = [1e21, 1e22, 1e23, 3e23]
labels = ['1e21 (小)', '1e22 (中)', '1e23 (大)', '3e23 (GPT-3级)']

print(f'{'计算预算':<18} {'最优参数量':>12} {'最优数据量':>14} {'token/param':>12} {'最低损失':>10}')
print('-' * 70)

results = []
for C, label in zip(compute_budgets, labels):
    N_opt, D_opt, L_opt = optimal_allocation(C)
    ratio = D_opt / N_opt
    results.append((C, N_opt, D_opt, L_opt))
    print(f'{label:<18} {N_opt/1e9:>10.1f}B {D_opt/1e9:>12.0f}B  {ratio:>10.1f}  {L_opt:>10.4f}')

print('\n关键发现（参数化模型 α=0.34, β=0.28）：最优 token/param 比值约 50–100。')
print('注：Hoffmann et al. 论文方法1/2（经验拟合）给出约 20 token/param。')
print('两种方法方向一致：数据量应远多于参数量；实践中常用 ~20 作为快速估算。')

## Part 4：验证「等比例增长」的规律

Chinchilla 论文的核心结论：计算量增加 10 倍时，N 和 D 都应增加约 √10 倍。

让我们用数据验证这个预测。

In [ ]:
# 在更宽的计算预算范围内验证等比例增长
C_range = np.logspace(19, 24, 20)
N_opts, D_opts = [], []

for C in C_range:
    N_opt, D_opt, _ = optimal_allocation(C)
    N_opts.append(N_opt)
    D_opts.append(D_opt)

N_opts = np.array(N_opts)
D_opts = np.array(D_opts)

# 用 log-log 线性回归估计幂指数
log_C = np.log10(C_range)
log_N = np.log10(N_opts)
log_D = np.log10(D_opts)

# 线性拟合：log_N = slope_N * log_C + intercept_N
slope_N = np.polyfit(log_C, log_N, 1)[0]
slope_D = np.polyfit(log_C, log_D, 1)[0]

fig, axes = plt.subplots(1, 2, figsize=(12, 4))

axes[0].loglog(C_range, N_opts, 'b-o', markersize=4, label=f'N_opt (slope={slope_N:.2f})')
axes[0].loglog(C_range, D_opts, 'r-s', markersize=4, label=f'D_opt (slope={slope_D:.2f})')
axes[0].set_xlabel('计算预算 C (FLOPs)')
axes[0].set_ylabel('最优值')
axes[0].set_title('最优 N 和 D 随计算量的变化')
axes[0].legend()
axes[0].grid(True, alpha=0.3, which='both')

# token/param 比例
axes[1].semilogx(C_range, D_opts / N_opts, 'g-o', markersize=4)
axes[1].axhline(y=20, color='orange', linestyle='--', label='经验规则 ~20（Hoffmann 方法1/2）')
axes[1].axhline(y=50, color='r', linestyle=':', label='参数化模型预测约 50–100')
axes[1].set_xlabel('计算预算 C (FLOPs)')
axes[1].set_ylabel('D_opt / N_opt（token/param）')
axes[1].set_title('最优 token/param 比例')
axes[1].legend()
axes[1].grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig(f'{ASSETS}/13-optimal-scaling.png', dpi=100, bbox_inches='tight')
plt.close()

print(f'N_opt 的幂指数 ≈ {slope_N:.3f}（理论值 0.5）')
print(f'D_opt 的幂指数 ≈ {slope_D:.3f}（理论值 0.5）')
print(f'平均 token/param 比 = {np.mean(D_opts/N_opts):.1f}')
print('\n结论：两者幂指数都接近 0.5，验证了等比例增长规律！')

## Part 5：Gopher vs Chinchilla 的直接对比

我们用模型预测两者的损失，验证「小模型充分训练能赢过大模型少量训练」。

In [ ]:
# 两个模型的配置
models = {
    'Gopher\n(280B, 300B tok)': {'N': 280e9, 'D': 300e9, 'color': '#e74c3c'},
    'Chinchilla\n(70B, 1.4T tok)': {'N': 70e9,  'D': 1400e9, 'color': '#2ecc71'},
    'GPT-3\n(175B, 300B tok)': {'N': 175e9, 'D': 300e9, 'color': '#3498db'},
    'LLaMA-7B\n(7B, 1T tok)': {'N': 7e9,   'D': 1000e9, 'color': '#f39c12'},
}

print('各模型对比（损失越小越好）:')
print('=' * 60)
losses = {}
for name, cfg in models.items():
    L = chinchilla_loss(cfg['N'], cfg['D'])
    losses[name] = L
    n_clean = name.replace('\n', ' ')
    print(f'{n_clean:<30} 损失 = {L:.4f}')

print()
print('排名（损失从低到高）:')
for rank, (name, L) in enumerate(
    sorted(losses.items(), key=lambda x: x[1]), 1
):
    n_clean = name.replace('\n', ' ')
    print(f'  第{rank}名: {n_clean}  ({L:.4f})')

# 可视化
fig, ax = plt.subplots(figsize=(8, 5))
names = list(models.keys())
vals  = [chinchilla_loss(m['N'], m['D']) for m in models.values()]
colors = [m['color'] for m in models.values()]
bars = ax.bar(names, vals, color=colors, edgecolor='black', alpha=0.85)
ax.set_ylabel('预测损失 L（越低越好）')
ax.set_title('不同模型的 Chinchilla 损失预测')
ax.set_ylim(min(vals)*0.98, max(vals)*1.02)
for bar, val in zip(bars, vals):
    ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.001,
            f'{val:.4f}', ha='center', va='bottom', fontsize=9)
plt.tight_layout()
plt.savefig(f'{ASSETS}/13-model-comparison.png', dpi=100, bbox_inches='tight')
plt.close()

chinchilla_L = chinchilla_loss(70e9, 1400e9)
gopher_L     = chinchilla_loss(280e9, 300e9)
print(f'\nChinchilla 损失 ({chinchilla_L:.4f}) < Gopher 损失 ({gopher_L:.4f}): {chinchilla_L < gopher_L}')

## Part 6：20 tokens/param 规则的实际意义

给定想训练的模型大小，可以快速算出需要多少训练数据。

In [ ]:
def compute_optimal_tokens(N_params):
    """给定参数量，用 Chinchilla 规律估算最优训练 token 数（~20x）"""
    return 20 * N_params

def compute_optimal_flops(N_params):
    """给定参数量，估算最优训练 FLOPs"""
    D = compute_optimal_tokens(N_params)
    return 6 * N_params * D

# 常见模型参数量
model_sizes = [
    ('7B (LLaMA-7B 级)',   7e9),
    ('13B (LLaMA-13B 级)', 13e9),
    ('70B (Chinchilla 级)', 70e9),
    ('175B (GPT-3 级)',    175e9),
    ('280B (Gopher 级)',   280e9),
]

print(f'{'模型':<25} {'参数量':>10} {'最优训练量':>14} {'最优FLOPs':>14}')
print('-' * 65)
for name, N in model_sizes:
    D_opt = compute_optimal_tokens(N)
    C_opt = compute_optimal_flops(N)
    print(f'{name:<25} {N/1e9:>8.0f}B {D_opt/1e9:>12.0f}B  {C_opt:.2e}')

print()
print('GPT-3 实际训练数据 300B tokens，而 Chinchilla 规律建议 175B 参数需要 3.5T tokens')
print(f'GPT-3 实际完成了最优训练量的 {300e9 / compute_optimal_tokens(175e9) * 100:.1f}%')

## 总结

本节手撕了 Chinchilla 缩放定律的核心数学：

| 概念 | 公式/数字 |
|------|----------|
| 幂律损失（单因子） | L(N) = A / N^α |
| 双因子损失 | L(N,D) = A/N^α + B/D^β + L∞ |
| 计算量约束 | C ≈ 6 × N × D |
| 最优缩放 | N_opt ∝ C^0.5，D_opt ∝ C^0.5 |
| 经验规则 | 每个参数约 20 token（论文方法1/2经验拟合）；参数化模型预测约 50–100 |
| 实验验证 | Chinchilla 70B < Gopher 280B（损失更低）|

---

**下一节**：节点14 — GPT-4 与涌现能力（即将推出）

*参考文献：Hoffmann et al. (2022) arXiv:2203.15556 | Kaplan et al. (2020) arXiv:2001.08361*